# SQL Aggregate Functions

This notebook covers:
- COUNT()
- SUM()
- AVG()
- MIN()
- MAX()
- GROUP BY
- HAVING
- Pandas Equivalent

# Load SQL Extension

In [1]:
!pip install ipython-sql psycopg2-binary


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
%load_ext sql

# Connect PostgreSQL

In [3]:
%sql postgresql://localhost/startup_commerce_db

# Create Employees Table

In [4]:
%%sql
DROP TABLE IF EXISTS employees;

CREATE TABLE employees (
    emp_id INT PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INT,
    city TEXT
);

 * postgresql://localhost/startup_commerce_db
Done.
Done.


[]

# Insert Sample Data

In [5]:
%%sql
INSERT INTO employees VALUES
(1,'Alice','Engineering',70000,'Delhi'),
(2,'Bob','HR',50000,'Mumbai'),
(3,'Charlie','Engineering',80000,'Delhi'),
(4,'David','Finance',60000,'Chennai'),
(5,'Eva','HR',55000,'Bangalore'),
(6,'Frank','Engineering',90000,'Mumbai'),
(7,'George','Finance',65000,'Delhi');

 * postgresql://localhost/startup_commerce_db
7 rows affected.


[]

# View Data

In [6]:
!pip install --upgrade "prettytable<3.0.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [7]:
%%sql
SELECT * FROM employees;

 * postgresql://localhost/startup_commerce_db
7 rows affected.


emp_id,name,department,salary,city
1,Alice,Engineering,70000,Delhi
2,Bob,HR,50000,Mumbai
3,Charlie,Engineering,80000,Delhi
4,David,Finance,60000,Chennai
5,Eva,HR,55000,Bangalore
6,Frank,Engineering,90000,Mumbai
7,George,Finance,65000,Delhi


# Convert SQL Result to Pandas

In [8]:
import pandas as pd

result = %sql SELECT * FROM employees;
df = result.DataFrame()
df

 * postgresql://localhost/startup_commerce_db
7 rows affected.


,emp_id,name,department,salary,city
0,1,Alice,Engineering,70000,Delhi
1,2,Bob,HR,50000,Mumbai
2,3,Charlie,Engineering,80000,Delhi
3,4,David,Finance,60000,Chennai
4,5,Eva,HR,55000,Bangalore
5,6,Frank,Engineering,90000,Mumbai
6,7,George,Finance,65000,Delhi


# COUNT() Function

In [9]:
%%sql
SELECT COUNT(*) AS total_employees
FROM employees;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


total_employees
7


In [10]:
len(df)

7

# SUM() Function

In [11]:
%%sql
SELECT SUM(salary) AS total_salary
FROM employees;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


total_salary
470000


In [12]:
df['salary'].sum()

np.int64(470000)

# AVG() Function

In [13]:
%%sql
SELECT AVG(salary) AS average_salary
FROM employees;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


average_salary
67142.857142857143


In [14]:
df['salary'].mean()

np.float64(67142.85714285714)

# MIN() Function

In [15]:
%%sql
SELECT MIN(salary) AS minimum_salary
FROM employees;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


minimum_salary
50000


In [16]:
df['salary'].min()

50000

# MAX() Function

In [17]:
%%sql
SELECT MAX(salary) AS maximum_salary
FROM employees;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


maximum_salary
90000


In [18]:
df['salary'].max()

90000

# GROUP BY

In [19]:
%%sql
SELECT department,
       COUNT(*) AS employees,
       AVG(salary) AS avg_salary
FROM employees
GROUP BY department;

 * postgresql://localhost/startup_commerce_db
3 rows affected.


department,employees,avg_salary
Finance,2,62500.000000000000
Engineering,3,80000.000000000000
HR,2,52500.000000000000


In [20]:
df.groupby('department').agg({
    'salary':['count','mean']
})

salary         
             count     mean
department                 
Engineering      3  80000.0
Finance          2  62500.0
HR               2  52500.0

# Multiple Aggregate Functions

In [21]:
%%sql
SELECT department,
       COUNT(*) AS employee_count,
       SUM(salary) AS total_salary,
       AVG(salary) AS avg_salary,
       MIN(salary) AS min_salary,
       MAX(salary) AS max_salary
FROM employees
GROUP BY department;

 * postgresql://localhost/startup_commerce_db
3 rows affected.


department,employee_count,total_salary,avg_salary,min_salary,max_salary
Finance,2,125000,62500.000000000000,60000,65000
Engineering,3,240000,80000.000000000000,70000,90000
HR,2,105000,52500.000000000000,50000,55000


In [22]:
df.groupby('department')['salary'].agg([
    'count',
    'sum',
    'mean',
    'min',
    'max'
])

,count,sum,mean,min,max
department,,,,,
Engineering,3,240000,80000.0,70000,90000
Finance,2,125000,62500.0,60000,65000
HR,2,105000,52500.0,50000,55000


# HAVING Clause

In [23]:
%%sql
SELECT department,
       AVG(salary) AS avg_salary
FROM employees
GROUP BY department
HAVING AVG(salary) > 65000;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


department,avg_salary
Engineering,80000.000000000000


In [24]:
grouped = df.groupby('department')['salary'].mean()
grouped[grouped > 65000]

department
Engineering    80000.0
Name: salary, dtype: float64

# Challenge 1

In [28]:
pd.DataFrame({
    'total_employees': [len(df)],
    'highest_salary': [df['salary'].max()],
    'lowest_salary': [df['salary'].min()]
})

,total_employees,highest_salary,lowest_salary
0,7,90000,50000


In [27]:
print("Total Employees:", len(df))
print("Highest Salary:", df['salary'].max())
print("Lowest Salary:", df['salary'].min())

Total Employees: 7
Highest Salary: 90000
Lowest Salary: 50000


In [26]:
%%sql
SELECT
    COUNT(*) AS total_employees,
    MAX(salary) AS highest_salary,
    MIN(salary) AS lowest_salary
FROM employees;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


total_employees,highest_salary,lowest_salary
7,90000,50000


# Challenge 2

In [29]:
%%sql
SELECT
    department,
    AVG(salary) AS average_salary
FROM employees
GROUP BY department;

 * postgresql://localhost/startup_commerce_db
3 rows affected.


department,average_salary
Finance,62500.000000000000
Engineering,80000.000000000000
HR,52500.000000000000


In [30]:
df.groupby('department')['salary'].mean()

department
Engineering    80000.0
Finance        62500.0
HR             52500.0
Name: salary, dtype: float64

In [31]:
df.groupby('department', as_index=False)['salary'].mean() \
  .rename(columns={'salary': 'average_salary'})

,department,average_salary
0,Engineering,80000.0
1,Finance,62500.0
2,HR,52500.0


# Challenge 3

In [32]:
%%sql
SELECT
    department,
    COUNT(*) AS employee_count
FROM employees
GROUP BY department
HAVING COUNT(*) > 2;

 * postgresql://localhost/startup_commerce_db
1 rows affected.


department,employee_count
Engineering,3


In [33]:
department_counts = df.groupby('department').size()

department_counts[department_counts > 2]

department
Engineering    3
dtype: int64

In [34]:
df.groupby('department').size().reset_index(name='employee_count').query('employee_count > 2')

,department,employee_count
0,Engineering,3


# Final Project

In [35]:
%%sql
SELECT
    department,
    COUNT(*) AS employee_count,
    SUM(salary) AS total_salary,
    AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY total_salary DESC;

 * postgresql://localhost/startup_commerce_db
3 rows affected.


department,employee_count,total_salary,avg_salary
Engineering,3,240000,80000.000000000000
Finance,2,125000,62500.000000000000
HR,2,105000,52500.000000000000


# SQL vs Pandas Cheat Sheet

| SQL | Pandas |
|------|---------|
| COUNT() | len(df) |
| SUM() | .sum() |
| AVG() | .mean() |
| MIN() | .min() |
| MAX() | .max() |
| GROUP BY | groupby() |
| HAVING | groupby()+filter |